# Fake News Detection using Bidirectional LSTM

This notebook is the reproducible analysis companion for the GitHub project. It classifies **short political claims** from the LIAR dataset into portfolio-level `Real` and `Fake` categories.

> **Responsible use:** This is an educational language-pattern classifier, not a fact-checking system. It does not retrieve evidence, verify sources, or establish objective truth.

## Rebuild highlights

- Uses the official LIAR train/validation/test splits.
- Converts dataset labels to names before binary mapping.
- Builds the vocabulary on training data only.
- Preserves selected punctuation and style signals.
- Uses a bidirectional LSTM with padding-aware mean/max pooling.
- Tunes the threshold on validation macro-F1.
- Compares against TF-IDF + logistic regression.
- Reports false positives, false negatives, ROC-AUC, and PR-AUC.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fake_news_pipeline import FakeNewsPipeline

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
print(PROJECT_ROOT)

## Inspect the packaged model card

In [ ]:
metadata = json.loads((MODELS_DIR / "model_metadata.json").read_text())
pd.Series({
    "Model": metadata["model_name"],
    "Scope": metadata["task_scope"],
    "Threshold": metadata["prediction_threshold"],
    "Vocabulary size": metadata["model_config"]["vocabulary_size"],
    "Maximum sequence length": metadata["tokenizer_config"]["maximum_sequence_length"],
})

## Load saved evaluation results

The repository already contains the evaluation produced from the held-out LIAR test split. Run `python train.py --source huggingface` from the project root to reproduce training.

In [ ]:
metrics_payload = json.loads((OUTPUTS_DIR / "model_metrics.json").read_text())
comparison = pd.DataFrame({
    "TF-IDF Logistic Regression": metrics_payload["tfidf_logistic_regression"]["test"],
    "Bidirectional LSTM": metrics_payload["lstm"]["test"],
}).T
comparison[["accuracy", "precision", "recall", "f1", "macro_f1", "roc_auc", "pr_auc"]].style.format("{:.3f}")

## Key evaluation figures

In [ ]:
from IPython.display import Image, display

for filename in [
    "class_distribution.png",
    "statement_length_distribution.png",
    "training_curve.png",
    "confusion_matrix.png",
    "roc_curve.png",
    "precision_recall_curve.png",
    "baseline_comparison.png",
]:
    print(filename)
    display(Image(filename=str(OUTPUTS_DIR / filename)))

## Error analysis

In [ ]:
errors = pd.read_csv(OUTPUTS_DIR / "error_analysis.csv")
errors.groupby("error_type").size().rename("count")

In [ ]:
errors[[
    "statement", "display_label", "predicted_label", "fake_probability", "confidence", "error_type"
]].head(15)

## Load the deployment pipeline

In [ ]:
pipeline = FakeNewsPipeline.load(MODELS_DIR)
result = pipeline.predict(
    "Officials published the complete budget proposal and supporting tables on the agency website."
)
result.to_dict()

## Batch inference example

In [ ]:
samples = pd.read_csv(PROJECT_ROOT / "data" / "sample_news.csv")
scored = pipeline.predict_batch(samples["text"])
pd.concat([samples, scored.drop(columns="text")], axis=1)

## Full retraining

Install training dependencies and execute:

```bash
pip install -r requirements-train.txt
python train.py --source huggingface
```

The training script regenerates the checkpoint, vocabulary, metadata, plots, metrics, sample predictions, and error analysis. For a local copy of the original TSV data, use the `--source local` options documented in the project README.

## Interpretation

The TF-IDF baseline slightly outperforms the LSTM on this benchmark. That is not a failure of the project; it is evidence that model selection should be empirical. LIAR statements are short, the dataset is modest in size, and sparse n-gram features remain strong. The LSTM implementation demonstrates sequence modeling while the comparison protects against overstating deep-learning value.